## Building a Database Agent with NexAU: SQL Tool and Skills Template

A database agent translates natural-language questions into SQL, executes the query, and returns a grounded answer. Building one that *works reliably* requires solving three problems:

1. **Safe SQL execution** — the model must only run read-only queries, with defense in depth.
2. **Domain knowledge injection** — the model needs to know that `price` is TEXT (not REAL), that cancelled orders must be excluded from revenue, etc.
3. **Scalability** — hand-writing knowledge docs for dozens of tables is impractical.

This notebook walks through a complete, reusable solution to all three:

| Component | What it does |
|---|---|
| `execute_sql` tool | Three-layer secure, read-only SQL executor with structured output |
| SKILL.md template | Per-table domain knowledge that teaches the model *how* to query correctly |
| `generate_skills` script | Auto-generates SKILL.md files from any SQLite database |

**Before/after example — why Skills matter:**

| Question | Without Skills | With Skills |
|---|---|---|
| "Which book is the most expensive?" | `ORDER BY price DESC` (TEXT sort — wrong) | `ORDER BY CAST(price AS REAL) DESC` (correct) |
| "Revenue in March?" | Includes cancelled orders (wrong) | Excludes `status = '已取消'` (correct) |
| "How many diamond members?" | Does not know the `member_level` column | Reads Skill, filters directly |

By the end, you will have a **reusable agent template** — swap in your own database and be running in 5 minutes.

### 1. Setup

#### 1.1 Install packages

This notebook only uses the Python standard library (`sqlite3`, `json`, `re`, `pathlib`). No extra packages are needed.

To run the full agent with NexAU afterward, install from the source repository (nexau is not yet published on PyPI):


In [ ]:
# nexau is not yet on PyPI — install from source.
# Uncomment and adjust the path if you have the nexau repo locally:
#   %pip install -e /path/to/nexau --quiet
# Or install from git:
#   %pip install git+https://github.com/YiRaaaan/nexau.git --quiet
%pip install python-dotenv --quiet

#### 1.2 Import modules

In [ ]:
from __future__ import annotations

import json
import os
import re
import sqlite3
import textwrap
import time
from pathlib import Path
from typing import Any

### 2. Create a sample database

We use a small **bookstore** scenario (3 tables) as a running example:

```
customers  <--+
              | customer_id
orders    ----+
              | book_id
books      <--+
```

The database is intentionally designed to include the most common pitfalls in real-world projects:

- `price` and `total_price` are **TEXT**, not REAL — sorting and aggregation require `CAST`.
- `status` includes `'已取消'` (cancelled) — revenue calculations must exclude these rows.
- `member_level` is a Chinese-language enum — there is no numeric rank column.

In [ ]:
DB_PATH = Path("sample_bookstore.sqlite")

if DB_PATH.exists():
    DB_PATH.unlink()

conn = sqlite3.connect(str(DB_PATH))
cur = conn.cursor()

# -- customers -------------------------------------------------------
cur.execute(
    "CREATE TABLE customers ("
    "  id          INTEGER PRIMARY KEY AUTOINCREMENT,"
    "  name        TEXT    NOT NULL,"
    "  email       TEXT    UNIQUE,"
    "  city        TEXT,"
    "  member_level TEXT   DEFAULT '普通',"
    "  created_at  TEXT    DEFAULT (datetime('now'))"
    ")"
)
customers = [
    ("张三", "zhangsan@example.com", "北京", "金卡"),
    ("李四", "lisi@example.com", "上海", "普通"),
    ("王五", "wangwu@example.com", "广州", "银卡"),
    ("赵六", "zhaoliu@example.com", "深圳", "钻石"),
    ("孙七", "sunqi@example.com", "杭州", "普通"),
    ("周八", "zhouba@example.com", "成都", "金卡"),
    ("吴九", "wujiu@example.com", "北京", "银卡"),
    ("郑十", "zhengshi@example.com", "上海", "普通"),
    ("钱十一", "qian11@example.com", "广州", "金卡"),
    ("陈十二", "chen12@example.com", "深圳", "普通"),
]
cur.executemany(
    "INSERT INTO customers (name, email, city, member_level) VALUES (?, ?, ?, ?)",
    customers,
)

# -- books -----------------------------------------------------------
cur.execute(
    "CREATE TABLE books ("
    "  id          INTEGER PRIMARY KEY AUTOINCREMENT,"
    "  title       TEXT    NOT NULL,"
    "  author      TEXT    NOT NULL,"
    "  genre       TEXT,"
    "  price       TEXT    NOT NULL,"
    "  stock       INTEGER DEFAULT 0,"
    "  publisher   TEXT,"
    "  publish_year INTEGER"
    ")"
)
books = [
    ("三体", "刘慈欣", "科幻", "36.00", 150, "重庆出版社", 2008),
    ("活着", "余华", "文学", "29.00", 200, "作家出版社", 1993),
    ("Python编程从入门到实践", "Eric Matthes", "技术", "89.00", 80, "人民邮电出版社", 2020),
    ("明朝那些事儿", "当年明月", "历史", "168.00", 60, "浙江人民出版社", 2009),
    ("原则", "瑞·达利欧", "经管", "98.00", 45, "中信出版社", 2018),
    ("流浪地球", "刘慈欣", "科幻", "32.00", 120, "中国华侨出版社", 2000),
    ("深度学习", "Ian Goodfellow", "技术", "168.00", 30, "人民邮电出版社", 2017),
    ("百年孤独", "马尔克斯", "文学", "55.00", 90, "南海出版公司", 2011),
    ("人类简史", "尤瓦尔·赫拉利", "历史", "68.00", 75, "中信出版社", 2014),
    ("从零到一", "彼得·蒂尔", "经管", "45.00", 55, "中信出版社", 2015),
    ("设计模式", "GoF", "技术", "79.00", 40, "机械工业出版社", 2000),
    ("围城", "钱钟书", "文学", "25.00", 180, "人民文学出版社", 1947),
]
cur.executemany(
    "INSERT INTO books (title, author, genre, price, stock, publisher, publish_year) "
    "VALUES (?, ?, ?, ?, ?, ?, ?)",
    books,
)

# -- orders ----------------------------------------------------------
cur.execute(
    "CREATE TABLE orders ("
    "  id          INTEGER PRIMARY KEY AUTOINCREMENT,"
    "  customer_id INTEGER NOT NULL REFERENCES customers(id),"
    "  book_id     INTEGER NOT NULL REFERENCES books(id),"
    "  quantity    INTEGER NOT NULL DEFAULT 1,"
    "  total_price TEXT    NOT NULL,"
    "  order_date  TEXT    NOT NULL,"
    "  status      TEXT    DEFAULT '已完成'"
    ")"
)
orders = [
    (1, 1, 2, "72.00", "2025-01-15", "已完成"),
    (1, 3, 1, "89.00", "2025-02-20", "已完成"),
    (2, 2, 1, "29.00", "2025-01-10", "已完成"),
    (2, 5, 1, "98.00", "2025-03-01", "待发货"),
    (3, 1, 1, "36.00", "2025-02-14", "已完成"),
    (3, 8, 2, "110.00", "2025-03-05", "已完成"),
    (4, 7, 1, "168.00", "2025-01-22", "已完成"),
    (4, 4, 1, "168.00", "2025-02-28", "已取消"),
    (5, 6, 3, "96.00", "2025-03-10", "待发货"),
    (6, 3, 1, "89.00", "2025-01-05", "已完成"),
    (6, 11, 1, "79.00", "2025-02-15", "已完成"),
    (7, 9, 1, "68.00", "2025-03-12", "已完成"),
    (8, 12, 1, "25.00", "2025-01-20", "已完成"),
    (9, 10, 2, "90.00", "2025-02-25", "待发货"),
    (10, 1, 1, "36.00", "2025-03-15", "已完成"),
]
cur.executemany(
    "INSERT INTO orders (customer_id, book_id, quantity, total_price, order_date, status) "
    "VALUES (?, ?, ?, ?, ?, ?)",
    orders,
)

conn.commit()
conn.close()

print(f"Database created: {DB_PATH}")
print(f"  Tables: customers ({len(customers)} rows), books ({len(books)} rows), orders ({len(orders)} rows)")

Quick sanity check — peek at the first few rows of each table:

In [ ]:
conn = sqlite3.connect(str(DB_PATH))
conn.row_factory = sqlite3.Row

for table in ["customers", "books", "orders"]:
    rows = conn.execute(f"SELECT * FROM {table} LIMIT 3").fetchall()
    print(f"\n-- {table} (first 3 rows) --")
    for r in rows:
        print(dict(r))

conn.close()

### 3. Build the `execute_sql` tool

`execute_sql` is the agent's only interface to the database. Two design goals:

**Three-layer security** — any single layer being bypassed is caught by the next:

```
           +-------------------------------+
 Layer 1   | Keyword whitelist + blacklist  |  Only SELECT / WITH allowed
           +-------------------------------+
 Layer 2   | SQL comment stripping          |  Prevents --foo\nDELETE bypass
           +-------------------------------+
 Layer 3   | file:...?mode=ro               |  SQLite engine-level read-only
           +-------------------------------+
```

**Structured output** — returns a JSON object (not a plain string) with `status`, `columns`, `data`, and `warnings`. The `warnings` field acts as hints to the model: suggest checking assumptions on empty results, or refining the query when results are truncated.

#### 3.1 Implementation

In [ ]:
MAX_ROWS = 10
MAX_OUTPUT_LENGTH = 50_000
DEFAULT_TIMEOUT = 30

_DANGEROUS_KEYWORDS = (
    "DROP", "TRUNCATE", "DELETE", "ALTER", "CREATE",
    "INSERT", "UPDATE", "REPLACE", "ATTACH", "DETACH",
    "PRAGMA", "VACUUM", "REINDEX", "GRANT", "REVOKE",
)


def _strip_sql_comments(sql: str) -> str:
    """Strip -- and /* */ comments to prevent injection bypass."""
    no_line = re.sub(r"--[^\n]*", "", sql)
    no_block = re.sub(r"/\*.*?\*/", "", no_line, flags=re.DOTALL)
    return no_block


def execute_sql(
    sql: str,
    db_path: str | Path = DB_PATH,
    timeout: int = DEFAULT_TIMEOUT,
    max_rows: int = MAX_ROWS,
) -> dict[str, Any]:
    """
    Execute a read-only SELECT query and return structured results.

    Returns a dict with keys:
      - status: "success" | "error" | "timeout"
      - columns, data, total_rows, row_count, truncated
      - warnings: list of hints for the model
    """
    start = time.time()

    # -- Layer 1: keyword check --
    if not sql or not sql.strip():
        return {"status": "error", "error": "SQL query cannot be empty",
                "duration_ms": int((time.time() - start) * 1000)}

    cleaned_upper = _strip_sql_comments(sql).strip().upper()
    for kw in _DANGEROUS_KEYWORDS:
        if re.match(rf"^{kw}(?:\s|$)", cleaned_upper):
            return {"status": "error",
                    "error": f"Only SELECT queries allowed. Found: {kw}",
                    "sql": sql,
                    "duration_ms": int((time.time() - start) * 1000)}

    if not re.match(r"^(SELECT|WITH)\b", cleaned_upper):
        return {"status": "error",
                "error": "Only SELECT or WITH ... SELECT queries allowed.",
                "sql": sql,
                "duration_ms": int((time.time() - start) * 1000)}

    # -- Layer 3: read-only connection --
    connection = None
    try:
        uri = f"file:{db_path}?mode=ro"
        connection = sqlite3.connect(uri, uri=True)
        connection.row_factory = sqlite3.Row

        deadline = time.time() + timeout
        connection.set_progress_handler(
            lambda: 1 if time.time() > deadline else 0, 1000
        )

        cursor = connection.execute(sql)
        col_names = [d[0] for d in cursor.description] if cursor.description else []

        # Use fetchmany instead of fetchall to avoid loading entire
        # result sets into memory on large tables without LIMIT.
        batch = cursor.fetchmany(max_rows + 1)
        truncated = len(batch) > max_rows
        rows_to_convert = batch[:max_rows]
        total_rows = max_rows + 1 if truncated else len(batch)

        data = [dict(r) for r in rows_to_convert]
        duration_ms = int((time.time() - start) * 1000)

        warnings: list[str] = []
        if total_rows == 0:
            warnings.append(
                "Query returned 0 rows. Check: table name, column names, "
                "WHERE conditions, data availability."
            )
        if truncated:
            warnings.append(
                f"Showing {max_rows} rows (more available). "
                f"Add WHERE clauses or LIMIT to narrow results."
            )

        result: dict[str, Any] = {
            "status": "success",
            "sql": sql,
            "columns": col_names,
            "data": data,
            "row_count": len(data),
            "total_rows": total_rows,
            "truncated": truncated,
            "duration_ms": duration_ms,
        }

        # Guard against individual rows with very large TEXT fields
        if len(json.dumps(result, ensure_ascii=False)) > MAX_OUTPUT_LENGTH:
            while (
                len(data) > 1
                and len(json.dumps(result, ensure_ascii=False)) > MAX_OUTPUT_LENGTH
            ):
                data = data[:-1]
                result["data"] = data
                result["row_count"] = len(data)
                result["truncated"] = True
            if not any("length limit" in w for w in warnings):
                warnings.append(
                    "Results truncated due to output length limit. "
                    "Select fewer columns or add WHERE clauses."
                )

        if warnings:
            result["warnings"] = warnings
        return result

    except sqlite3.OperationalError as e:
        msg = str(e)
        if "interrupted" in msg.lower():
            return {"status": "timeout",
                    "error": f"Query timed out after {timeout}s",
                    "sql": sql,
                    "duration_ms": int((time.time() - start) * 1000)}
        return {"status": "error", "error": msg, "sql": sql,
                "duration_ms": int((time.time() - start) * 1000)}
    except Exception as e:
        return {"status": "error", "error": str(e), "sql": sql,
                "duration_ms": int((time.time() - start) * 1000)}
    finally:
        if connection:
            connection.close()

#### 3.2 Test the tool

Let's verify both normal queries and security rejection:

In [ ]:
# Normal query
result = execute_sql("SELECT title, price FROM books LIMIT 3")
print(json.dumps(result, ensure_ascii=False, indent=2))

In [ ]:
# Empty result triggers a warning hint for the model
result = execute_sql("SELECT * FROM books WHERE genre = '玄幻'")
print(json.dumps(result, ensure_ascii=False, indent=2))

In [ ]:
# Security: reject DROP
result = execute_sql("DROP TABLE books")
print(json.dumps(result, ensure_ascii=False, indent=2))

In [ ]:
# Security: reject comment-based bypass
result = execute_sql("-- just a comment\nDELETE FROM books")
print(json.dumps(result, ensure_ascii=False, indent=2))

#### 3.3 The TEXT-as-number pitfall

This is the single most common mistake models make when querying databases. When `price` is stored as TEXT, sorting gives lexicographic order (`"89.00" > "168.00"`) instead of numeric order. Compare:

In [ ]:
# Wrong: TEXT sort — "89.00" comes before "168.00" lexicographically
result = execute_sql("SELECT title, price FROM books ORDER BY price DESC LIMIT 5")
print("TEXT sort (wrong):")
for row in result["data"]:
    print(f"  {row['title']:20s} {row['price']}")

print()

# Correct: CAST to REAL for numeric ordering
result = execute_sql("SELECT title, CAST(price AS REAL) AS price FROM books ORDER BY price DESC LIMIT 5")
print("Numeric sort (correct):")
for row in result["data"]:
    print(f"  {row['title']:20s} {row['price']}")

### 4. Write SKILL.md — domain knowledge for database tables

Skills are NexAU's mechanism for injecting domain knowledge into the model. **Each database table gets its own Skill** — the model reads a table's SKILL.md before querying it, just as a human would consult a data dictionary before writing SQL.

#### 4.1 SKILL.md structure

A well-written SKILL.md contains six sections:

| Section | Purpose | Key principle |
|---|---|---|
| `description` (frontmatter) | Routing — the model uses this to decide *whether* to read the Skill | Use specific trigger words, not "Information about X" |
| **When to use** | Positive routing: example questions | 3-5 representative questions |
| **Schema** | Column names, types, PK, FK, semantics | The model's "data dictionary" |
| **Common values** | All possible values for enum columns | Model can reference directly instead of guessing |
| **Example queries** | Few-shot SQL examples | Model will follow these patterns |
| **Gotchas** | Pitfalls and caveats | **The most valuable section** — directly reduces errors |

> **Gotchas is the most valuable section.** Models don't usually fail because they "don't know the table exists" — they fail because they don't know `price` is TEXT, or that cancelled orders must be excluded.

#### 4.2 Full example: SKILL.md for the `books` table

In [ ]:
books_skill = '''---
name: books
description: >-
  Use this skill whenever the user asks about books — titles, authors, genres,
  prices, stock levels, or publishers. Join to orders via book_id.
---

# books

The complete book catalog. One row per book, keyed by `id`.

## When to use

- "What science fiction books do we have?"
- "Which book is the most expensive?"
- "How many books by 刘慈欣?"

## Schema

| Column | Type | Description |
|---|---|---|
| `id` | INTEGER PK | Book ID — join key for `orders.book_id` |
| `title` | TEXT | Book title |
| `author` | TEXT | Author name |
| `genre` | TEXT | Category: `文学` / `技术` / `历史` / `科幻` / `经管` |
| `price` | TEXT | Price in yuan — **TEXT not numeric**, use `CAST(price AS REAL)` |
| `stock` | INTEGER | Inventory count |
| `publisher` | TEXT | Publisher name |
| `publish_year` | INTEGER | Year of publication |

## Common values

- `genre`: `文学`, `技术`, `历史`, `科幻`, `经管`

## Example queries

**Most expensive books:**

```sql
SELECT title, author, CAST(price AS REAL) AS price_yuan
FROM books
ORDER BY price_yuan DESC
LIMIT 5;
```

## Gotchas

- `price` is **TEXT**, not REAL — always `CAST(price AS REAL)` for numeric ops.
- `genre` uses Chinese category names. For fuzzy search use `LIKE '%技术%'`.
'''

print(books_skill)

#### 4.3 Anti-patterns to avoid

| Anti-pattern | Consequence |
|---|---|
| `description` too vague | Model routes incorrectly — reads when it shouldn't, skips when it should |
| No Gotchas section | Model repeatedly hits TEXT-as-number sorting, missing status filters, etc. |
| Example queries use `SELECT *` | Model copies the pattern — result sets become too large |
| All knowledge in system prompt | Wastes tokens every turn, dilutes model attention |
| One Skill for multiple tables | Routing granularity too coarse — can't load on demand |

### 5. Auto-generate Skills

Hand-writing SKILL.md is ideal for fine-tuning, but impractical for databases with dozens of tables. The script below connects to any SQLite database and **auto-generates** a SKILL.md draft for each table.

It automatically detects:

- Table structure (column names, types, PK, FK)
- TEXT columns storing numeric values (common pitfall, e.g., price fields)
- Enum-like columns with limited distinct values
- FK relationships with auto-generated JOIN examples

Sections that require human review are marked with `[TODO]`.

#### 5.1 Helper functions

In [ ]:
def get_tables(conn: sqlite3.Connection) -> list[str]:
    """Return all user table names."""
    rows = conn.execute(
        "SELECT name FROM sqlite_master WHERE type='table' "
        "AND name NOT LIKE 'sqlite_%' ORDER BY name"
    ).fetchall()
    return [r[0] for r in rows]


def get_table_info(conn: sqlite3.Connection, table: str) -> list[dict]:
    """Return column metadata for a table."""
    # Table names come from sqlite_master, so they are trusted identifiers.
    rows = conn.execute(f"PRAGMA table_info('{table}')").fetchall()
    return [{"name": r[1], "type": r[2] or "TEXT", "pk": bool(r[5]),
             "notnull": bool(r[3]), "default": r[4]} for r in rows]


def get_foreign_keys(conn: sqlite3.Connection, table: str) -> dict[str, str]:
    """Return FK mapping {local_col: 'ref_table.ref_col'}."""
    rows = conn.execute(f"PRAGMA foreign_key_list('{table}')").fetchall()
    return {r[3]: f"{r[2]}.{r[4]}" for r in rows}


def get_common_values(conn: sqlite3.Connection, table: str, col: str,
                      limit: int = 8) -> list[tuple[str, int]]:
    """Return the most common values in a column."""
    try:
        rows = conn.execute(
            f"SELECT [{col}], COUNT(*) AS n FROM [{table}] "
            f"WHERE [{col}] IS NOT NULL "
            f"GROUP BY [{col}] ORDER BY n DESC LIMIT ?", (limit,)
        ).fetchall()
        return [(str(r[0]), r[1]) for r in rows]
    except Exception:
        return []


def is_numeric_in_text(conn: sqlite3.Connection, table: str, col: str) -> bool:
    """Detect if a TEXT column actually stores numeric values."""
    try:
        sample = conn.execute(
            f"SELECT [{col}] FROM [{table}] WHERE [{col}] IS NOT NULL LIMIT 20"
        ).fetchall()
        if not sample:
            return False
        numeric_count = sum(
            1 for r in sample
            if r[0] is not None and re.match(r"^-?\d+(\.\d+)?$", str(r[0]).strip())
        )
        return numeric_count / len(sample) > 0.8
    except Exception:
        return False

#### 5.2 Main generator function

In [ ]:
def generate_skill_md(conn: sqlite3.Connection, table: str) -> str:
    """Generate a SKILL.md draft for a single table."""
    columns = get_table_info(conn, table)
    fk_map = get_foreign_keys(conn, table)
    row_count = conn.execute(f"SELECT COUNT(*) FROM [{table}]").fetchone()[0]

    pk_cols = [c["name"] for c in columns if c["pk"]]
    pk_str = ", ".join(f"`{c}`" for c in pk_cols) if pk_cols else "(no explicit PK)"

    # FK notes
    fk_notes = [f"`{lc}` -> `{ref}`" for lc, ref in fk_map.items()]
    fk_hint = " Join keys: " + ", ".join(fk_notes) + "." if fk_map else ""

    # Schema table
    schema_rows = []
    for col in columns:
        parts = []
        if col["pk"]:
            parts.append("PK")
        if col["name"] in fk_map:
            parts.append(f"FK -> `{fk_map[col['name']]}`")
        if col["notnull"] and not col["pk"]:
            parts.append("NOT NULL")
        if col["default"] is not None:
            parts.append(f"default: {col['default']}")
        desc = " | ".join(parts) if parts else ""
        schema_rows.append(f"| `{col['name']}` | {col['type']} | {desc} |")

    # Common values (enum-like TEXT columns only)
    common_sections = []
    for col in columns:
        if col["type"].upper() not in ("TEXT",) or col["pk"]:
            continue
        vals = get_common_values(conn, table, col["name"])
        if 2 <= len(vals) <= 10:
            val_str = ", ".join(f"`{v[0]}`" for v in vals)
            common_sections.append(f"- `{col['name']}`: {val_str}")

    # Gotchas
    gotchas = []
    for col in columns:
        if col["type"].upper() == "TEXT" and is_numeric_in_text(conn, table, col["name"]):
            gotchas.append(
                f"- `{col['name']}` is **TEXT** but stores numeric values — "
                f"use `CAST({col['name']} AS REAL)` for comparisons and sorting."
            )

    # Example queries
    select_cols = ", ".join(f"[{c['name']}]" for c in columns[:5])
    examples = f"**Browse:**\n\n```sql\nSELECT {select_cols}\nFROM [{table}]\nLIMIT 10;\n```"

    if fk_map:
        fk_col, ref = next(iter(fk_map.items()))
        ref_table, ref_col = ref.split(".")
        examples += (
            f"\n\n**Join with `{ref_table}`:**\n\n```sql\n"
            f"SELECT t.*, r.*\nFROM [{table}] t\n"
            f"JOIN [{ref_table}] r ON t.[{fk_col}] = r.[{ref_col}]\n"
            f"LIMIT 10;\n```"
        )

    nl = "\n"
    return (f"---\nname: {table}\ndescription: >-\n"
            f"  Use this skill whenever the user asks about data in the `{table}` table.{fk_hint}\n"
            f"  [TODO: Add routing keywords]\n---\n\n"
            f"# {table}\n\nRows: **{row_count}** | PK: {pk_str}\n"
            + (f"Foreign keys: {', '.join(fk_notes)}\n" if fk_notes else "")
            + f"\n## When to use\n\n- [TODO: Add 3-5 example questions]\n"
            f"\n## Schema\n\n| Column | Type | Description |\n|---|---|---|\n"
            f"{nl.join(schema_rows)}\n"
            f"\n## Common values\n\n"
            f"{nl.join(common_sections) if common_sections else '- [TODO: Add common values]'}\n"
            f"\n## Example queries\n\n{examples}\n"
            f"\n## Gotchas\n\n"
            f"{nl.join(gotchas) if gotchas else '- [TODO: Add known pitfalls]'}\n")

#### 5.3 Run on the sample database

In [ ]:
conn = sqlite3.connect(str(DB_PATH))
tables = get_tables(conn)
print(f"Found {len(tables)} tables: {', '.join(tables)}\n")

for table in tables:
    print("=" * 60)
    print(f"  {table}")
    print("=" * 60)
    print(generate_skill_md(conn, table))

conn.close()

Note what the generator detected automatically:

- `books.price` and `orders.total_price` correctly identified as "TEXT storing numbers"
- Foreign key relationships on `orders` detected, with JOIN examples generated
- Enum columns (`status`, `genre`, `member_level`) had their common values listed
- `[TODO]` markers highlight what needs human review

### 6. Tool Schema definitions

NexAU defines tool schemas via YAML — this is the "instruction manual" the model sees when deciding how to call a tool.

#### 6.1 ExecuteSQL.tool.yaml

```yaml
type: tool
name: execute_sql
description: >-
  Execute a read-only SQL query against the connected database.
  Only SELECT queries are allowed. Results are capped at max_rows rows.
  Always use LIMIT. For large tables, run COUNT(*) first.

input_schema:
  type: object
  properties:
    sql:
      type: string
      description: The SQL query to execute (SELECT only).
    timeout:
      type: integer
      default: 30
    max_rows:
      type: integer
      default: 10
  required: [sql]
```

> **Key insight:** Best practices in the `description` ("Always use LIMIT") directly influence model behavior. Putting guidance in the tool schema is more effective than the system prompt — the model re-reads it on every tool-calling decision.

#### 6.2 TodoWrite.tool.yaml

```yaml
type: tool
name: write_todos
description: >-
  Maintain a structured task list for multi-step queries.
  Use when the question requires 2+ tables or multiple queries.

input_schema:
  type: object
  properties:
    todos:
      type: array
      items:
        type: object
        properties:
          id: { type: string }
          content: { type: string }
          status: { type: string, enum: [pending, in_progress, completed] }
        required: [id, content, status]
  required: [todos]
```

`write_todos` serves as the model's scratchpad — for cross-table queries, it lists steps first and executes them one by one. It uses NexAU's built-in implementation, no Python code required.

### 7. System Prompt design

The system prompt defines the agent's behavior pattern. The core is a **7-step workflow**:

```
Plan -> Track tasks -> Read Skills -> Write SQL -> Execute -> Reflect -> Answer
```

In [ ]:
SYSTEM_PROMPT_TEMPLATE = """
You are a database agent. Your job: translate natural-language questions
into correct SQL, execute the query, and return a clear answer grounded in the data.

## Database

- Engine: SQLite (read-only via `execute_sql`)
- Tables: {table_list}
- Primary join keys: {join_keys}

**Detailed schema and example queries for each table are provided as Skills —
one Skill per table. ALWAYS read the relevant Skill before writing a query.**

## Workflow

1. **Plan.** Identify which tables you need.
2. **Track tasks (when complex).** If the question requires 2+ tables OR multiple
   queries, call `write_todos` to record one task per step.
3. **Read Skills.** For every table you plan to touch, read its Skill first.
   Pay special attention to the **Gotchas** section.
4. **Write the SQL.** SQLite syntax. Always `LIMIT`. Prefer explicit columns.
5. **Execute.** Call `execute_sql`.
6. **Reflect.** If `total_rows == 0` or `warnings` is present, re-read the Skill
   and try a different query.
7. **Answer** in the user's language with a concise answer. End with the SQL
   in a fenced block.

## Constraints

- The tool rejects any non-SELECT statement — don't try.
- No hallucinated columns. If the user asks about a column that doesn't exist,
  say so explicitly.
"""

system_prompt = SYSTEM_PROMPT_TEMPLATE.format(
    table_list="`customers`, `books`, `orders`",
    join_keys="`orders.customer_id` -> `customers.id`, `orders.book_id` -> `books.id`",
)

print(system_prompt)

**Key design decisions:**

- **"ALWAYS read the relevant Skill"** — without this, models skip Skill reading and guess column names.
- **Track tasks only for complex questions** — simple single-table queries skip this step to avoid overhead.
- **Reflect step** — when the model sees empty results or warnings, it must reconsider rather than immediately answering "no data found".

### 8. Assemble the Agent

With all components ready, `agent.yaml` ties them together:

```yaml
type: agent
name: database_agent
description: A general-purpose agent that answers questions over a SQL database.

system_prompt: ./system_prompt.md
system_prompt_type: file
max_iterations: 50
tool_call_mode: structured

llm_config:
  model: ${env.LLM_MODEL}
  base_url: ${env.LLM_BASE_URL}
  api_key: ${env.LLM_API_KEY}
  temperature: 0.2
  stream: true

tools:
  - name: execute_sql
    yaml_path: ./tools/ExecuteSQL.tool.yaml
    binding: tools.execute_sql:execute_sql

  - name: write_todos
    yaml_path: ./tools/TodoWrite.tool.yaml
    binding: nexau.archs.tool.builtin.session_tools:write_todos

skills:
  - ./skills/customers
  - ./skills/books
  - ./skills/orders
```

**File structure:**

```
my_db_agent/
├── agent.yaml
├── system_prompt.md
├── tools/
│   ├── ExecuteSQL.tool.yaml
│   ├── execute_sql.py
│   └── TodoWrite.tool.yaml
└── skills/
    ├── customers/SKILL.md
    ├── books/SKILL.md
    └── orders/SKILL.md
```

### 9. End-to-end test

We can simulate the agent's behavior directly with `execute_sql` — writing SQL as the Skill would guide it. No NexAU runtime needed.

#### 9.1 Simple query: "Which book is the most expensive?"

The agent reads the `books` Skill, discovers `price` is TEXT, and uses `CAST`:

In [ ]:
result = execute_sql(
    "SELECT title, author, CAST(price AS REAL) AS price "
    "FROM books ORDER BY price DESC LIMIT 3"
)
print("Q: Which book is the most expensive?")
print(f"A: {result['data'][0]['title']}, priced at {result['data'][0]['price']} yuan")
print(f"SQL: {result['sql']}")

#### 9.2 Filtered aggregation: "Revenue in March 2025?"

The agent reads the `orders` Skill, learns to exclude cancelled orders and CAST `total_price`:

In [ ]:
result = execute_sql(
    "SELECT SUM(CAST(total_price AS REAL)) AS revenue "
    "FROM orders "
    "WHERE order_date >= '2025-03-01' AND order_date < '2025-04-01' "
    "  AND status != '已取消'"
)
print("Q: Revenue in March 2025?")
print(f"A: {result['data'][0]['revenue']} yuan")
print(f"SQL: {result['sql']}")

#### 9.3 Cross-table JOIN: "Which customer spent the most?"

The agent reads both `orders` and `customers` Skills, then JOINs:

In [ ]:
result = execute_sql(
    "SELECT c.name, COUNT(*) AS orders, "
    "  SUM(CAST(o.total_price AS REAL)) AS total_spent "
    "FROM orders o "
    "JOIN customers c ON o.customer_id = c.id "
    "WHERE o.status != '已取消' "
    "GROUP BY c.id ORDER BY total_spent DESC LIMIT 5"
)
print("Q: Which customer spent the most?")
for row in result["data"]:
    print(f"  {row['name']:8s}  {row['orders']} orders  {row['total_spent']} yuan")
print(f"\nSQL: {result['sql']}")

#### 9.4 Error recovery: the Reflect step in action

A key part of the 7-step workflow is **Reflect** — when the model gets empty results or errors, it should re-read the Skill and retry. Let's simulate this:

In [ ]:
# Attempt 1: The model guesses a column name that doesn't exist
print("--- Attempt 1: model guesses a column name ---")
result = execute_sql("SELECT name, level FROM customers LIMIT 5")
print(f"Status: {result['status']}")
print(f"Error: {result.get('error', 'none')}")

print()

# Attempt 2: Model reads the Skill, discovers the column is called 'member_level'
print("--- Attempt 2: after reading the customers Skill ---")
result = execute_sql("SELECT name, member_level FROM customers LIMIT 5")
print(f"Status: {result['status']}")
print(f"Data: {result['data']}")

print()

# Attempt 3: Empty result triggers reflection
print("--- Attempt 3: query returns 0 rows ---")
result = execute_sql("SELECT * FROM customers WHERE city = '武汉'")
print(f"Status: {result['status']}, rows: {result['total_rows']}")
print(f"Warnings: {result.get('warnings', [])}")
print()
print("The model sees the warning and adjusts — perhaps the city doesn't exist,")
print("or the filter should be broader.")

#### 9.5 Validate Skills: run example queries

A good practice after writing Skills is to **run all example queries** and verify they succeed. This catches typos in column names, incorrect CAST patterns, or stale schema references:

In [ ]:
skill_example_queries = {
    "books: most expensive": (
        "SELECT title, author, CAST(price AS REAL) AS price_yuan "
        "FROM books ORDER BY price_yuan DESC LIMIT 5"
    ),
    "orders: monthly revenue": (
        "SELECT strftime('%Y-%m', order_date) AS month, "
        "SUM(CAST(total_price AS REAL)) AS revenue "
        "FROM orders WHERE status != '已取消' GROUP BY month ORDER BY month"
    ),
    "orders: top customers (JOIN)": (
        "SELECT c.name, COUNT(*) AS order_count, "
        "SUM(CAST(o.total_price AS REAL)) AS total_spent "
        "FROM orders o JOIN customers c ON o.customer_id = c.id "
        "WHERE o.status != '已取消' GROUP BY c.id ORDER BY total_spent DESC LIMIT 5"
    ),
}

print("Validating Skill example queries:\n")
for name, sql in skill_example_queries.items():
    result = execute_sql(sql)
    status = result['status']
    rows = result.get('row_count', 0)
    duration = result.get('duration_ms', 0)
    marker = "PASS" if status == "success" and rows > 0 else "FAIL"
    print(f"  [{marker}] {name}: {status}, {rows} rows, {duration}ms")

### 10. Summary and next steps

This notebook implemented the three core components for building a database agent:

| Component | Value |
|---|---|
| `execute_sql` tool | Three-layer security + structured output + warnings that guide model self-correction |
| SKILL.md template | One-table-one-Skill best practice; Gotchas directly reduces model errors |
| `generate_skills` script | Automates 0-to-80%; human review takes it to 95% |

**Next steps:**

1. **Plug in your own database** — replace `DB_PATH`, run `generate_skills`, fill in the `[TODO]` markers.
2. **Configure a NexAU agent** — use the template files in `template/`.
3. **Read the full tutorial** — [Chapter 2: Custom SQL Tool](../zh/02-custom-tool.md) and [Chapter 3: Skills](../zh/03-skills.md).
4. **Adapt for other databases** — swap `sqlite3` for `psycopg2` / `pymysql`, replace `PRAGMA table_info` with `information_schema` queries.

| File to modify | What to change |
|---|---|
| `execute_sql.py` | Replace `sqlite3` with target driver (`psycopg2`, `pymysql`) |
| `ExecuteSQL.tool.yaml` | Update SQL dialect notes in `description` |
| `system_prompt.md` | Update "Engine: SQLite" to target database |
| `generate_skills` | Replace `PRAGMA table_info` with `information_schema` queries |

In [ ]:
# Cleanup: remove the sample database
if DB_PATH.exists():
    DB_PATH.unlink()
    print(f"Cleaned up {DB_PATH}")